In [6]:
import csv
import math
from pathlib import Path


def csv_to_latex_table(
    csv_path: str | Path,
    caption: str = "",
    label: str = "",
    decimal_places: int = 4,
    highlight_best: bool = True,
    best_is: str = "min",          # "min" | "max"
    col_rename: dict[str, str] | None = None,
    row_rename: dict[str, str] | None = None,
) -> str:
    """
    Convert a CSV file of numeric results to a LaTeX table.

    Parameters
    ----------
    csv_path : path-like
        Path to the CSV file. The first column is treated as a row-label
        column; all remaining columns must be numeric.
    caption : str
        Text for \\caption{}.  Omitted if empty.
    label : str
        Text for \\label{}.  Omitted if empty.
    decimal_places : int
        Number of decimal places passed to Python's round() before
        LaTeX receives the value.  LaTeX itself does no additional
        rounding.
    highlight_best : bool
        If True, the best value in each numeric column is typeset in
        bold via \\textbf{}.
    best_is : "min" | "max"
        Which extremum counts as "best".
    col_rename : dict[str, str] | None
        Optional mapping {raw_header -> display_header}.
    row_rename : dict[str, str] | None
        Optional mapping {raw_row_label -> display_row_label}.

    Returns
    -------
    str
        A complete LaTeX table environment as a string.
    """
    csv_path = Path(csv_path)
    col_rename = col_rename or {}
    row_rename = row_rename or {}

    # ── 1. Parse CSV ────────────────────────────────────────────────────────
    with csv_path.open(newline="") as fh:
        reader = csv.DictReader(fh)
        fieldnames = reader.fieldnames or []
        rows = list(reader)

    if not fieldnames:
        raise ValueError(f"No header found in {csv_path}")

    row_col, *num_cols = fieldnames

    # Convert to floats
    data: list[dict[str, float | str]] = []
    for row in rows:
        entry: dict[str, float | str] = {row_col: row[row_col]}
        for col in num_cols:
            entry[col] = float(row[col])
        data.append(entry)

    # ── 2. Find best value per numeric column ───────────────────────────────
    best: dict[str, float] = {}
    if highlight_best:
        selector = min if best_is == "min" else max
        for col in num_cols:
            best[col] = selector(float(r[col]) for r in data)  # type: ignore[arg-type]

    # ── 3. Format a single numeric cell ─────────────────────────────────────
    def fmt(value: float, col: str) -> str:
        rounded = round(value, decimal_places)
        text = f"{rounded:.{decimal_places}f}"
        if highlight_best and math.isclose(value, best[col], rel_tol=1e-9):
            return rf"\textbf{{{text}}}"
        return text

    # ── 4. Build header row ──────────────────────────────────────────────────
    def display_col(c: str) -> str:
        return col_rename.get(c, c).replace("_", r"\_")

    header_cells = [display_col(row_col)] + [display_col(c) for c in num_cols]
    header_line = " & ".join(header_cells) + r" \\"

    # ── 5. Build data rows ───────────────────────────────────────────────────
    body_lines: list[str] = []
    for entry in data:
        row_label = row_rename.get(str(entry[row_col]), str(entry[row_col])).replace("_", " ")
        cells = [row_label] + [fmt(float(entry[c]), c) for c in num_cols]  # type: ignore[arg-type]
        body_lines.append(" & ".join(cells) + r" \\")

    # ── 6. Column spec: l for label, c for each numeric column ──────────────
    col_spec = "l" + "c" * len(num_cols)

    # ── 7. Assemble table environment ───────────────────────────────────────
    lines: list[str] = [
        r"\begin{table}[tb]",
        r"  \centering",
    ]
    if caption:
        lines.append(rf"  \caption{{{caption}}}")
    if label:
        lines.append(rf"  \label{{{label}}}")
    lines += [
        r"   \resizebox{\columnwidth}{!}{%",
        rf"  \begin{{tabular}}{{{col_spec}}}",
        r"    \toprule",
        f"    {header_line}",
        r"    \midrule",
    ]
    for bl in body_lines:
        lines.append(f"    {bl}")
    lines += [
        r"    \bottomrule",
        r"  \end{tabular}",
        r"   }",
        r"\end{table}",
    ]

    return "\n".join(lines)


# ── Convenience: batch-convert a list of CSV files ──────────────────────────
def csvs_to_latex(
    csv_paths: list[str | Path],
    output_path: str | Path | None = None,
    **kwargs,
) -> str:
    """
    Convert multiple CSV files to LaTeX tables and concatenate them.

    Parameters
    ----------
    csv_paths : list of path-like
        Input CSV files.  Each becomes one table.
    output_path : path-like or None
        If given, the combined output is written here.
    **kwargs
        Forwarded to csv_to_latex_table for every file.

    Returns
    -------
    str
        All tables joined by blank lines.
    """
    tables = []
    for path in csv_paths:
        path = Path(path)
        kw = dict(kwargs)
        kw.setdefault("caption", path.stem.replace("_", " ").title())
        kw.setdefault("label", f"tab:{path.stem}")
        tables.append(csv_to_latex_table(path, **kw))

    combined = "\n\n".join(tables)

    if output_path is not None:
        Path(output_path).write_text(combined)

    return combined

In [7]:
latex = csv_to_latex_table(
    #"stats_mse_by_distribution.csv",
    #"stats_mse_by_payoff.csv",
    "stats_errors.csv",
    caption="MSE comparison across distributions",
    label="tab:mse_comparison",
    decimal_places=4,
    highlight_best=False,
    best_is="min",
    col_rename={
        "dist_name":              "Distribution",
        "gpq_mse":               "GPQ",
        "hsgp_mse":              "HSGP",
        "quantum_analytical_mse": "QA-HSGP (analytical)",
        "quantum_mse":           "QA-HSGP",
    },
)
print(latex)

\begin{table}[tb]
  \centering
  \caption{MSE comparison across distributions}
  \label{tab:mse_comparison}
   \resizebox{\columnwidth}{!}{%
  \begin{tabular}{lcccccccc}
    \toprule
    method & mean\_pct\_err & median\_pct\_err & max\_pct\_err & min\_pct\_err & mean\_mse & median\_mse & max\_mse & min\_mse \\
    \midrule
    gpq & 15.3081 & 5.1352 & 99.9448 & 0.2818 & 0.2513 & 0.0209 & 2.6909 & 0.0049 \\
    hsgp & 23.8301 & 4.9227 & 194.4465 & 0.2686 & 0.7330 & 0.0225 & 8.7106 & 0.0047 \\
    quantum analytical & 23.8300 & 4.9228 & 194.4462 & 0.2688 & 0.7330 & 0.0225 & 8.7106 & 0.0047 \\
    quantum & 5.8647 & 2.7531 & 30.4723 & 0.1031 & 0.0858 & 0.0225 & 0.7891 & 0.0044 \\
    \bottomrule
  \end{tabular}
   }
\end{table}
